In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# (Step 1) Create the quantitative predictive model
    1. Create price only baseline
    2. Add technical features + fundamentals

    Below is model based solely on price vvvvvv

In [4]:
def build_price_dataset(ticker, start="2010-01-01", end=None):
    df = yf.download(ticker, start=start, end=end, interval="1d", auto_adjust=False).reset_index()
    df = df.rename(columns={"Date": "date"})
    
    # returns
    df["ret"] = df["Adj Close"].pct_change()
    df["log_ret"] = np.log(df["Adj Close"]).diff()

    # rolling stats (past-only)
    df["ma_5"] = df["Adj Close"].rolling(5).mean()
    df["ma_10"] = df["Adj Close"].rolling(10).mean()
    df["vol_5"] = df["ret"].rolling(5).std()
    df["vol_10"] = df["ret"].rolling(10).std()

    # momentum
    df["mom_5"] = df["Adj Close"] / df["Adj Close"].shift(5) - 1
    df["mom_10"] = df["Adj Close"] / df["Adj Close"].shift(10) - 1

    # volume features
    df["vol_chg"] = df["Volume"].pct_change()
    df["vol_ma_5"] = df["Volume"].rolling(5).mean()

    # label: next-day direction
    df["next_ret"] = df["ret"].shift(-1)
    df["y"] = (df["next_ret"] > 0).astype(int)

    df = df.dropna().reset_index(drop=True)
    return df

In [6]:
def train_test_one_ticker(df, feature_cols):
    split = int(len(df) * 0.7)  # chronological split
    train = df.iloc[:split]
    test  = df.iloc[split:]

    X_train, y_train = train[feature_cols], train["y"]
    X_test, y_test   = test[feature_cols],  test["y"]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=3000))
    ])

    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    return {
        "n_train": len(train),
        "n_test": len(test),
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
    }

In [8]:
tickers = ["AAPL", "AMZN", "NFLX", "GOOGL", "META"]

features = [
    "ret","log_ret","ma_5","ma_10","vol_5","vol_10",
    "mom_5","mom_10","vol_chg","vol_ma_5"
]

results = []
for tkr in tickers:
    df = build_price_dataset(tkr, start="2010-01-01")
    metrics = train_test_one_ticker(df, features)
    metrics["ticker"] = tkr
    results.append(metrics)

results_df = pd.DataFrame(results).set_index("ticker").sort_index()
results_df

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,n_train,n_test,accuracy,f1,roc_auc
ticker,,,,,
AAPL,2837,1217,0.512736,0.665539,0.460896
AMZN,2837,1217,0.510271,0.675027,0.498422
GOOGL,2837,1217,0.535744,0.695746,0.493751
META,2418,1037,0.514947,0.581879,0.510189
NFLX,2837,1217,0.493837,0.380282,0.501892


### Model + technical data

# (Step 2) This below is focused on adding sentiment to our current predictive quantitative model

In [11]:
# Arrange Y-M-D so that dataset has most recent data at top
news = pd.read_csv("abcnews-date-text.csv")
news["date"] = pd.to_datetime(news["publish_date"].astype(str), format="%Y%m%d").dt.date
news["headline_text"] = news["headline_text"].astype(str)

news.head() # View data

,publish_date,headline_text,date
0,20030219,aba decides against community broadcasting lic...,2003-02-19
1,20030219,act fire witnesses must be aware of defamation,2003-02-19
2,20030219,a g calls for infrastructure protection summit,2003-02-19
3,20030219,air nz staff in aust strike for pay rise,2003-02-19
4,20030219,air nz strike to affect australian travellers,2003-02-19


In [17]:
# Apply word filters on headlines for FAANG
company_terms = {
    "AAPL": ["apple", "iphone", "ipad", "mac", "tim cook"],
    "AMZN": ["amazon", "aws", "prime", "bezos"],
    "NFLX": ["netflix", "streaming"],
    "GOOGL": ["google", "alphabet", "youtube", "android", "sundar pichai"],
    "META": ["meta", "facebook", "instagram", "whatsapp", "zuckerberg"],
}

def make_pattern(words):
    return re.compile(r"(?i)\b(?:%s)\b" % "|".join(map(re.escape, words)))

for tkr, words in company_terms.items():
    news[f"m_{tkr}"] = news["headline_text"].str.contains(make_pattern(words), na=False)

{'AAPL': 930, 'AMZN': 1443, 'NFLX': 670, 'GOOGL': 785, 'META': 1441}